# Understanding BERT and GPT

Two Transformer-based models have stood out for their effectiveness and versatility: BERT (Bidirectional Encoder Representations from Transformers) and GPT (Generative Pretrained Transformer). Both models have pushed the boundaries of what's possible in NLP, achieving state-of-the-art results on a range of tasks.

## BERT (Bidirectional Encoder Representations from Transformers)

### Introduction to BERT

BERT is a transformer-based model designed to understand the context of a word in search queries and other text by looking at the words that come before and after it. This is known as bidirectional context, which is a key differentiator between BERT and GPT.

### Architecture of BERT

BERT is based on the Transformer architecture, which uses self-attention mechanisms. It consists of multiple layers of bidirectional transformers. BERT comes in two sizes, BERT Base and BERT Large, with the latter having more transformer blocks (layers), more attention heads, and a larger hidden layer size.

![bert.png](./assets/bert.png)

### Pre-training and Fine-tuning

BERT is pre-trained on a large corpus of text in two unsupervised tasks: Masked Language Modeling (MLM) and Next Sentence Prediction (NSP). In MLM, some percentage of the input tokens are masked at random, and then the model attempts to predict the masked words, based on their context. In NSP, the model learns to predict whether two sentences are likely to be consecutive.

After pre-training, BERT can be fine-tuned with additional output layers for a wide range of tasks without substantial task-specific architecture modifications.

![mlm](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/MLM.png)

![mask-input.png](./assets/mask-input.png)

![Input-Embedding.png](./assets/Input-Embedding.png)

![Encoder-Layer-0.png](./assets/Encoder-Layer-0.png)

![Encoder-Layer-1.png](./assets/Encoder-Layer-1.png)

![Output-layer.png](./assets/Output-layer.png)

![Prediction.png](./assets/Prediction.png)

> see https://www.101ai.net/text/bert for interactive guide to vizualize through the variables, epochs, data samples and data elements.

### Applications of BERT

BERT has been successfully applied to many NLP tasks, including:

- Text classification
- Question answering
> - Sentiment analysis
> - Named entity recognition

### Sentiment Analysis with BERT

Sentiment analysis is the task of classifying the polarity of a given text, determining whether the expressed opinion is positive, negative, or neutral. BERT has been highly successful in sentiment analysis tasks due to their deep understanding of language context.

We'll use the Hugging Face `transformers` library to fine-tune a pre-trained BERT model for sentiment analysis on the IMDB dataset, which contains 50,000 movie reviews labeled as positive or negative.

In [ ]:
!pip install transformers[torch]

In [ ]:
!pip install datasets transformers[torch]

In [ ]:
from datasets import load_dataset

# Load the IMDB dataset
# dataset = load_dataset("imdb")
dataset = load_dataset("stanfordnlp/imdb")
# Split the dataset into training and testing sets
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
len(train_dataset)

In [ ]:
train_dataset[2]

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments # download from huggingface
import torch

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')#lower case everything, APPLE = apple
 #bert-base-cased, keep all casing upper case = emotion? MIT (mit) , NUS, Named Entity Recognition , APPLE, apple

# Tokenize the dataset
def preprocess_function(examples): # max of 512 context window
    return tokenizer(examples["text"], 
                     truncation=True, 
                     padding=True, 
                     max_length=128 # defining it <= 512 because no need so much context for the dataset
                     )

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)

In [ ]:
# Convert the labels into binary format
def convert_labels(example):
    if example['label'] == 1:
        example['label'] = 1  # Positive sentiment
    else:
        example['label'] = 0  # Negative sentiment 0,1,2
    return example

binary_train_dataset = tokenized_train_dataset.map(convert_labels)
binary_test_dataset = tokenized_test_dataset.map(convert_labels)

In [ ]:
# Load the pre-trained BERT model for sequence classification layer
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', 
                                                      num_labels=2 # 2 means positive and negative, binary classification
                                                      )

In [ ]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,             # 1 epoch for faster training
    per_device_train_batch_size=16, # if have 2 GPU set up; 32
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=binary_train_dataset,
    eval_dataset=binary_test_dataset,
)

# Train the model
trainer.train()

In [ ]:
# Evaluate the model
results = trainer.evaluate()
print(results)

validation loss < training loss, model may be underfit

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1)
    return "positive" if prediction.item() == 1 else "negative"

In [ ]:
# Test on a new review
new_review = "The movie was fantastic! I loved it."
print(predict_sentiment(new_review))

In [ ]:
# Test on another review
new_review = "The movie was not very great."
print(predict_sentiment(new_review))

In [ ]:
# Test on another review
new_review = "i rather watch paint dry"
print(predict_sentiment(new_review))

In [ ]:
# Test on another review
new_review = "i had a good nap"
print(predict_sentiment(new_review))

### Named Entity Recognition (NER) with BERT

Named Entity Recognition (NER) is a subtask of information extraction that seeks to locate and classify named entities mentioned in unstructured text into predefined categories such as the names of persons, organizations, locations, expressions of times, quantities, monetary values, percentages, etc.

For example, in the sentence "Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in Cupertino, California, on April 1, 1976," a NER system would identify "Apple Inc." as an organization, "Steve Jobs," "Steve Wozniak," and "Ronald Wayne" as persons, "Cupertino, California" as a location, and "April 1, 1976" as a date.

We'll use pre-trained BERT model for Named Entity Recognition (NER).

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

In [ ]:
# Load a pre-trained model and tokenizer
# for model_name, just paste the model name from huggingface
model_name = "dbmdz/bert-large-cased-finetuned-conll03-english"
model = AutoModelForTokenClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Encode text
text = "Hugging Face Inc. is a company based in New York City. Its headquarters are in DUMBO, therefore very close to the Manhattan Bridge."
inputs = tokenizer.encode(text, return_tensors="pt")

In [ ]:
# Predict NER tags
with torch.no_grad():
    outputs = model(inputs).logits
predictions = torch.argmax(outputs, dim=2)

In [ ]:
# Convert predictions to labels
labels = [model.config.id2label[prediction] for prediction in predictions[0].tolist()]

In [ ]:
# Print the tokens with their predicted NER tags
for token, label in zip(tokenizer.tokenize(text), labels):
    print(f"{token}: {label}")

## GPT (Generative Pretrained Transformer)

### Introduction to GPT

GPT, like BERT, is based on the Transformer model, but unlike BERT, GPT is unidirectional and is designed to generate text. GPT takes an input of text and continues to generate text based on this input.

![bert-vs-gpt](https://www.researchgate.net/publication/354908597/figure/fig1/AS:1073191841722368@1632880282219/Model-structure-of-BERT-and-GPT.png)

### Architecture of GPT

The architecture of GPT is similar to the decoder part of the Transformer model. It uses masked self-attention, where each token can only attend to previous tokens in the self-attention layers of the transformer.

![masked-self-attention.png](./assets/masked-self-attention.png)

### Pre-training and Fine-tuning

GPT is also pre-trained on a large corpus of text in an unsupervised manner. It uses a task called Language Modeling, where the model predicts the next word in a sentence given the previous words. After pre-training, GPT can be fine-tuned on a specific task by adding an additional output layer and continuing the training process on a labeled dataset.

### Applications of GPT

GPT has been used for applications such as:

> - Text generation
- Conversational agents
- Language translation
- Text summarization

Both BERT and GPT have their strengths and are chosen based on the requirements of the task. BERT's bidirectional context is powerful for understanding the meaning of words within their context, making it ideal for tasks that require understanding the relationship between words. GPT's strength lies in generating coherent and contextually relevant text, making it suitable for tasks that involve text continuation.

These models have paved the way for more advanced language models like GPT-2, GPT-3, and BERT-based models like RoBERTa and DistilBERT, each offering improvements and variations that make them suited for different NLP tasks.

### Text Generation with GPT-2

Let's use a pre-trained GPT-2 model to generate text. Hugging Face provides several versions of GPT-2 that are already trained on a diverse corpus of text data.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [ ]:
# Load pre-trained GPT-2 model and tokenizer
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

In [ ]:
def preprocess(text):
    # Encode the input text
    return tokenizer.encode(text, return_tensors='pt', add_special_tokens=True)

def generate(text, max_length=150):
    # Preprocess the text
    input_ids = preprocess(text)

    # Generate
    summary_ids = model.generate(input_ids,
                                 max_length=max_length,
                                 num_beams=5, #beam search greedy decoding
                                 # sampling
                                 no_repeat_ngram_size=2,
                                 early_stopping=True)

    # Decode the text
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
text_to_generate = """
The history of natural language processing (NLP) generally started in the 1950s, although work can be found from earlier periods.
In 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence.
"""

generated = generate(text_to_generate)
print(generated)

In [ ]:
text_to_generate = """
The quick brown fox
"""

generated = generate(text_to_generate)
print(generated)